In [1]:
import os
import json
import pickle

# 1️⃣ JSON 파일 읽기
json_path = '/root/Public_Storage/madelab_khw/lpcv/img_correct.json'
with open(json_path, 'r') as f:
    dic = json.load(f)  # ✅ `read(f)` → `json.load(f)`

# 2️⃣ pickle 파일 경로 설정 (절대 경로 사용)
path = '/root/Public_Storage/madelab_khw/lpcv/key_img_correct.pickle'

# 3️⃣ 파일 존재 여부 확인 후 처리
if os.path.exists(path):
    print('s')  # ✅ 파일이 존재하면 's' 출력
else:
    # ✅ 빈 파일 생성
    open(path, 'wb').close()

    # ✅ pickle 파일로 저장 (딕셔너리 키를 리스트로 변환 후 저장)
    with open(path, 'wb') as f:
        pickle.dump(list(dic.keys()), f)  # ✅ JSON 딕셔너리 키를 pickle 형식으로 저장


s


In [2]:
print(dic.keys())

dict_keys(['Bicycle', 'Car', 'Motorcycle', 'Airplane', 'Bus', 'Train', 'Truck', 'Boat', 'Traffic_Light', 'Stop_Sign', 'Parking_Meter', 'Bench', 'Bird', 'Cat', 'Dog', 'Horse', 'Sheep', 'Cow', 'Elephant', 'Bear', 'Zebra', 'Backpack', 'Umbrella', 'Handbag', 'Tie', 'Skis', 'Sports_Ball', 'Kite', 'Tennis_Racket', 'Bottle', 'Wine Glass', 'Cup', 'Knife', 'Spoon', 'Bowl', 'Banana', 'Apple', 'Orange', 'Broccoli', 'Hot_Dog', 'Pizza', 'Donut', 'Chair', 'Couch', 'Potted_Plant', 'Bed', 'Dining_Table', 'Toilet', 'TV', 'Laptop', 'Mouse', 'Remote', 'Keyboard', 'Cell_Phone', 'Microwave', 'Oven', 'Toaster', 'Sink', 'Refrigerator', 'Book', 'Clock', 'Vase', 'Teddy_Bear', 'Hair_Drier'])


In [3]:
import os
import json
import requests
import cv2
from pycocotools.coco import COCO
from tqdm import tqdm

with open('/root/Public_Storage/madelab_khw/lpcv/key_img_correct.pickle', 'rb') as f:
    categoryname = pickle.load(f)

print(len(categoryname))

64


In [8]:
annotation_dir = "/root/Public_Storage/madelab_khw/lpcv/coco/annotations/"
os.makedirs(annotation_dir, exist_ok = True)

train_json = os.path.join(annotation_dir, "annotations/instances_train2017.json")
valid_json = os.path.join(annotation_dir, "annotations/instances_val2017.json")
train_url = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"

In [5]:
if not os.path.exists(train_json) or os.path.exists(valid_json):
    print(f"COCO 데이터셋이 없습니다. 다운로드 중...")
    os.system(f"wget {train_url} -P {annotation_dir} && unzip {annotation_dir}/annotations_trainval2017.zip -d {annotation_dir}")
    print("다운로드 완료!")

COCO 데이터셋이 없습니다. 다운로드 중...


--2025-03-10 09:32:13--  http://images.cocodataset.org/annotations/annotations_trainval2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 52.217.121.217, 3.5.25.164, 3.5.3.165, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|52.217.121.217|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 252907541 (241M) [application/zip]
Saving to: ‘/root/Public_Storage/madelab_khw/lpcv/coco/annotations/annotations_trainval2017.zip’

     0K .......... .......... .......... .......... ..........  0%  128K 32m9s
    50K .......... .......... .......... .......... ..........  0% 9.21M 16m17s
   100K .......... .......... .......... .......... ..........  0%  262K 16m6s
   150K .......... .......... .......... .......... ..........  0% 64.4M 12m5s
   200K .......... .......... .......... .......... ..........  0%  257K 12m52s
   250K .......... .......... .......... .......... ..........  0% 49.2M 10m44s
   300K .......... .......... ..........

Archive:  /root/Public_Storage/madelab_khw/lpcv/coco/annotations//annotations_trainval2017.zip
  inflating: /root/Public_Storage/madelab_khw/lpcv/coco/annotations/annotations/instances_train2017.json  
  inflating: /root/Public_Storage/madelab_khw/lpcv/coco/annotations/annotations/instances_val2017.json  
  inflating: /root/Public_Storage/madelab_khw/lpcv/coco/annotations/annotations/captions_train2017.json  
  inflating: /root/Public_Storage/madelab_khw/lpcv/coco/annotations/annotations/captions_val2017.json  
  inflating: /root/Public_Storage/madelab_khw/lpcv/coco/annotations/annotations/person_keypoints_train2017.json  
  inflating: /root/Public_Storage/madelab_khw/lpcv/coco/annotations/annotations/person_keypoints_val2017.json  
다운로드 완료!


In [9]:
coco_train = COCO(train_json)
coco_test= COCO(valid_json)

catIds = coco_train.getCatIds(catNms=categoryname)

loading annotations into memory...
Done (t=13.46s)
creating index...
index created!
loading annotations into memory...
Done (t=0.41s)
creating index...
index created!


In [10]:
import re
from pycocotools.coco import COCO


# COCO 데이터셋 로드
coco = COCO(train_json)
categories = coco.loadCats(coco.getCatIds())
coco_category_names = {cat['name'].lower(): cat['name'] for cat in categories} 
categoryname_fixed = []


for cat in categoryname:
    fixed_name = re.sub(r'[_]', ' ', cat).lower()  # '_' 제거 후 소문자로 변환
    if fixed_name in coco_category_names:
        categoryname_fixed.append(coco_category_names[fixed_name])
    else:
        print(f"⚠ '{cat}'이(가) COCO 데이터셋에 존재하지 않음!")

print(f"✅ 변환된 COCO 클래스 이름 목록: {categoryname_fixed}")

# 🚀 COCO에서 올바른 클래스 ID 가져오기
catIds = coco.getCatIds(catNms=categoryname_fixed)
if not catIds:
    print(f"🚨 Error: '{categoryname_fixed}'에 해당하는 COCO 카테고리가 없습니다!")
else:
    print(f"✅ 선택된 카테고리 ID = {catIds}")


loading annotations into memory...
Done (t=14.07s)
creating index...
index created!
✅ 변환된 COCO 클래스 이름 목록: ['bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'backpack', 'umbrella', 'handbag', 'tie', 'skis', 'sports ball', 'kite', 'tennis racket', 'bottle', 'wine glass', 'cup', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'orange', 'broccoli', 'hot dog', 'pizza', 'donut', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'teddy bear', 'hair drier']
✅ 선택된 카테고리 ID = [2, 3, 4, 5, 6, 7, 8, 9, 10, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 27, 28, 31, 32, 35, 37, 38, 43, 44, 46, 47, 49, 50, 51, 52, 53, 55, 56, 58, 59, 60, 62, 63, 64, 65, 67, 70, 72, 73, 74, 75, 76, 77, 78, 7

In [ ]:
import os
import json
import requests
import cv2
import re
from tqdm import tqdm
from pycocotools.coco import COCO


# 7️⃣ COCO에서 해당 카테고리에 포함된 이미지 ID 가져오기 (getAnnIds 방식)
annIds_train = coco_train.getAnnIds(catIds=catIds)
annIds_test = coco_test.getAnnIds(catIds=catIds)

imgIds_train = list(set([ann["image_id"] for ann in coco_train.loadAnns(annIds_train)]))
imgIds_test = list(set([ann["image_id"] for ann in coco_test.loadAnns(annIds_test)]))

print(f"✅ Train 이미지 개수: {len(imgIds_train)}")
print(f"✅ Test 이미지 개수: {len(imgIds_test)}")

if len(imgIds_train) == 0 and len(imgIds_test) == 0:
    print("🚨 Error: 선택한 카테고리에 해당하는 이미지가 COCO 데이터셋에 없습니다.")
    exit()

# 8️⃣ 이미지 ID를 사용하여 이미지 정보 가져오기
images_train = coco_train.loadImgs(imgIds_train)
images_test = coco_test.loadImgs(imgIds_test)

# 9️⃣ 저장할 폴더 생성 (절대 경로)
filtered_dir_train = "/root/Public_Storage/madelab_khw/lpcv/coco/filtered_dir_train"
filtered_dir_test = "/root/Public_Storage/madelab_khw/lpcv/coco/filtered_dir_test"

cropped_dir_train = "/root/Public_Storage/madelab_khw/lpcv/coco/cropped_dir_train"
cropped_dir_test = "/root/Public_Storage/madelab_khw/lpcv/coco/cropped_dir_test"

for directory in [filtered_dir_train, filtered_dir_test, cropped_dir_train, cropped_dir_test]:
    os.makedirs(directory, exist_ok=True)

# 10️⃣ 이미지 다운로드 함수
def download_images(images, save_dir):
    for img in tqdm(images, desc=f"📥 Downloading images to {save_dir}"):
        try:
            img_url = img["coco_url"]
            img_data = requests.get(img_url).content
            img_path = os.path.join(save_dir, img["file_name"])

            with open(img_path, "wb") as f:
                f.write(img_data)

        except KeyError:
            print(f"⚠ Warning: 'coco_url' 키가 없는 이미지입니다. (ID: {img['id']})")
            continue  # 오류 발생 시 다음 이미지로 넘어감

download_images(images_train, filtered_dir_train)
download_images(images_test, filtered_dir_test)

print(f"✅ Train/Test 데이터에서 선택된 카테고리 이미지 다운로드 완료!")

# 11️⃣ 선택된 카테고리 객체만 크롭하는 함수
def crop_objects(images, coco, save_dir):
    for img in tqdm(images, desc=f"✂ Cropping objects in {save_dir}"):
        img_path = os.path.join(save_dir, img["file_name"])

        # OpenCV로 이미지 로드
        image = cv2.imread(img_path)

        # 이미지가 제대로 로드되지 않은 경우 건너뛰기
        if image is None:
            print(f"⚠ Warning: {img['file_name']}을(를) 로드할 수 없습니다. 건너뜁니다.")
            continue

        # 해당 이미지의 바운딩 박스 정보 가져오기 (선택된 카테고리만)
        annIds = coco.getAnnIds(imgIds=[img["id"]], catIds=catIds, iscrowd=None)
        anns = coco.loadAnns(annIds)

        # 객체 크롭 및 저장
        for i, ann in enumerate(anns):
            x, y, w, h = map(int, ann["bbox"])  # 바운딩 박스 좌표
            cropped_img = image[y:y+h, x:x+w]  # Crop

            # 저장할 경로
            crop_path = os.path.join(save_dir, f"{img['file_name'].split('.')[0]}_obj_{i}.jpg")

            # 크롭된 이미지 저장
            cv2.imwrite(crop_path, cropped_img)

crop_objects(images_train, coco_train, cropped_dir_train)
crop_objects(images_test, coco_test, cropped_dir_test)

print(f"✅ Train/Test 데이터에서 선택된 카테고리 객체 크롭 완료!")


✅ Train 이미지 개수: 104877
✅ Test 이미지 개수: 4453


📥 Downloading images to /root/Public_Storage/madelab_khw/lpcv/coco/filtered_dir_train:   5% 5530/104877 [1:51:39<30:52:30,  1.12s/it]  IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

📥 Downloading images to /root/Public_Storage/madelab_khw/lpcv/coco/filtered_dir_train: 100% 104877/104877 [33:39:47<00:00,  1.16s/it]   
📥 Downloading images to /root/Public_Storage/madelab_khw/lpcv/coco/filtered_dir_test:  85% 3767/4453 [1:13:25<11:48,  1.03s/it]  